# W2D2 Motor-RNN Connectivity Experiment

**Primary question:** Under variance-normalized initialization and a fixed recurrent learning rule, how does recurrent connection probability $p$ affect held-out learning of a six-direction center-out reaching task?

This is the current reproducible analysis notebook. It keeps the motor decoder fixed during training and evaluation; PCA/manifold dimensionality is secondary and exploratory. When opened from GitHub in Google Colab, the setup cell clones and installs the repository automatically.


In [ ]:
# Repository-aware setup for local Jupyter and Google Colab.
import os
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = os.environ.get(
    "NMA_MOTOR_RNN_REPOSITORY",
    "https://github.com/Thom-320/nma-motor-rnn-connectivity.git",
)

def find_repository_root():
    candidates = [Path.cwd(), Path.cwd().parent]
    return next(
        (path for path in candidates if (path / "src/nma_motor_rnn").exists()),
        None,
    )

repository_root = find_repository_root()
if repository_root is None and "google.colab" in sys.modules:
    checkout = Path("nma-motor-rnn-connectivity")
    if not checkout.exists():
        subprocess.run(["git", "clone", REPOSITORY_URL, str(checkout)], check=True)
    repository_root = checkout.resolve()
    os.chdir(repository_root)

if repository_root is None:
    raise FileNotFoundError(
        "Run Jupyter from the repository root, or set NMA_MOTOR_RNN_REPOSITORY in Colab."
    )

sys.path.insert(0, str((repository_root / "src").resolve()))

import pandas as pd
from nma_motor_rnn.connectivity import (
    hypothesis_contrasts, pilot_config, plot_required_figures,
    primary_config, run_experiment, write_rows_csv,
)


## Scientific commitments

- H1: final held-out NMSE at $p=0.05$ exceeds the mean at $p=0.10,0.20,0.40$.
- H2: improvement from $p=0.05$ to $0.20$ exceeds improvement from $p=0.20$ to $0.40$.
- Positive seed-level contrasts support the corresponding hypothesis.
- $D_{PR}$, $D_{90}$, and dimension-performance associations are exploratory.
- The existing $p=0.1$ versus $0.5$ online-loss plot is not included as confirmatory evidence.

In [ ]:
# Pilot: N=100, three paired seeds, four densities, 40 trials.
pilot = pilot_config()
pilot_checkpoints, pilot_conditions, pilot_seconds = run_experiment(
    pilot, seeds=range(3), output_dir='results/pilot'
)
print(f'Pilot runtime: {pilot_seconds / 60:.2f} minutes')
pd.DataFrame(pilot_conditions)


In [ ]:
# Pilot figures and preregistered seed-level contrasts.
pilot_figures = plot_required_figures(
    pilot_checkpoints, pilot_conditions, 'results/pilot/figures'
)
pilot_contrasts = hypothesis_contrasts(pilot_conditions)
write_rows_csv('results/pilot/hypothesis_contrasts.csv', pilot_contrasts)
display(pd.DataFrame(pilot_contrasts))
pilot_figures


## Primary experiment

The preregistered runtime rule uses eight seeds if the complete pilot took under 30 minutes and five seeds otherwise. Do not change this rule after inspecting which seed count gives a preferred result.

In [ ]:
primary = primary_config()
n_primary_seeds = 8 if pilot_seconds < 30 * 60 else 5
print(f'Running N=200 primary experiment with {n_primary_seeds} paired seeds')
primary_checkpoints, primary_conditions, primary_seconds = run_experiment(
    primary, seeds=range(n_primary_seeds), output_dir='results/primary'
)
print(f'Primary runtime: {primary_seconds / 60:.2f} minutes')
pd.DataFrame(primary_conditions)


In [ ]:
primary_figures = plot_required_figures(
    primary_checkpoints, primary_conditions, 'results/primary/figures'
)
primary_contrasts = hypothesis_contrasts(primary_conditions)
write_rows_csv('results/primary/hypothesis_contrasts.csv', primary_contrasts)
contrast_df = pd.DataFrame(primary_contrasts)
display(contrast_df)
display(contrast_df[['h1_contrast', 'h2_contrast']].agg(['mean', 'std']))
primary_figures


## Reading the result without overclaiming

1. First inspect held-out NMSE and learning-curve AUC across independent network seeds.
2. Positive mean H1/H2 contrasts are directionally consistent with the hypotheses; show the seed-level variation.
3. Online feedback loss is diagnostic only and must not replace held-out velocity error.
4. Interpret manifold measurements as possible correlates, not established mechanisms.
5. If performance is similar across densities, report a robustness/null result within the tested range.
6. The manipulation changes density and the number of plastic weights together; do not claim density independent of plasticity budget.